# H&M Dataset — FOAF Decentralized MF Experiment

Mirrors the ML-100K experiment structure exactly:
- Same FOAF gradient-sharing method (user-stratified, random-sampling batch)
- Same graph topologies: Random-2-out, Random-5-out, Scale-Free, Cycle
- Same evaluation metric: RMSE on held-out test set
- Same baselines: CL (centralized), FL (FedAvg), GL (Gossip), LL (local)

**Data source**: Download `articles.csv`, `customers.csv`, `transactions_train.csv`  
from https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations

**Target**: purchase frequency `y_ij = count(customer_i, product_j)` — same as appendix.

## 0. Imports

In [3]:
import pandas as pd
import numpy as np
import datetime
import math
import copy
import os
from dataclasses import dataclass, field
from typing import Dict, Optional
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import networkx as nx
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

SEED = 42

## 1. Data Loading & Preprocessing (follows H_M_Data.ipynb)

In [5]:
# ── Load raw files ──────────────────────────────────────────────────────────
DATA_DIR = "."   # change to path containing the three CSVs

df           = pd.read_csv(os.path.join(DATA_DIR, "transactions_train.csv"),
                           dtype={"article_id": str})
customer_df  = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"),
                           dtype={"article_id": str, "product_code": str})
article_df   = pd.read_csv(os.path.join(DATA_DIR, "articles.csv"),
                           dtype={"article_id": str})

df["product_code"] = df["article_id"].map(
    article_df.set_index("article_id")["product_code"])

# ── Week index (0 = most recent week) ───────────────────────────────────────
df["t_dat"] = pd.to_datetime(df["t_dat"])
df["week"]  = (df["t_dat"].max() - df["t_dat"]).dt.days // 7

print("Raw transactions:", len(df))

Raw transactions: 31788324


In [6]:
# ── Filter: top-6 product types, ≥120-purchase customers, top-5000 locations
# (exactly as in appendix Section 3.2 and H_M_Data.ipynb)
chosen_types = (
    article_df.groupby("product_type_name")["product_code"]
    .count().sort_values()[-6:].index
)
chosen_customers = (
    df.groupby("customer_id")["product_code"].nunique()
    .pipe(lambda s: s[s > 120]).index
)
location_customers = customer_df["postal_code"].value_counts()[1:5001].index
customer_df = customer_df[customer_df["postal_code"].isin(location_customers)]

chosen_articles = article_df[article_df["product_type_name"].isin(chosen_types)]
df = df[df["product_code"].isin(chosen_articles["product_code"])]
df = df[df["customer_id"].isin(chosen_customers)]
df = df[df["customer_id"].isin(customer_df["customer_id"])]

df.drop(["t_dat", "price", "article_id", "sales_channel_id"], axis=1, inplace=True)
df["bought"] = 1
df.reset_index(drop=True, inplace=True)

print("Filtered transactions:", len(df))

Filtered transactions: 216390


In [7]:
# ── Temporal split: week > 50 → train history, week ≤ 50 → test
# (week=0 is most recent; lower week number = more recent)
TEST_WEEK_CUTOFF = 50   # most recent 50 weeks = test set

train_raw = df[df["week"] >  TEST_WEEK_CUTOFF].copy()   # older weeks = train
test_raw  = df[df["week"] <= TEST_WEEK_CUTOFF].copy()   # recent weeks = test

# Aggregate to (customer, product) → purchase count
train_df = (
    train_raw.groupby(["customer_id", "product_code"])["bought"]
    .count().reset_index()
)
test_df = (
    test_raw.groupby(["customer_id", "product_code"])["bought"]
    .count().reset_index()
)

# Drop test entries whose customer or item is not in train
train_customers = set(train_df["customer_id"].unique())
train_items     = set(train_df["product_code"].unique())
test_df = test_df[
    test_df["customer_id"].isin(train_customers) &
    test_df["product_code"].isin(train_items)
].copy()

# ── Integer encoding ─────────────────────────────────────────────────────────
customer_id2idx = {c: i for i, c in enumerate(train_df["customer_id"].unique())}
item_id2idx     = {a: i for i, a in enumerate(train_df["product_code"].unique())}

for df_ in [train_df, test_df]:
    df_["customer_id"]  = df_["customer_id"].map(customer_id2idx)
    df_["product_code"] = df_["product_code"].map(item_id2idx)
    df_.dropna(inplace=True)  # remove unmapped test entries
    df_[["customer_id", "product_code"]] = df_[["customer_id", "product_code"]].astype(int)

N_USERS = train_df["customer_id"].nunique()
N_ITEMS = train_df["product_code"].nunique()

print(f"Users: {N_USERS} | Items: {N_ITEMS}")
print(f"Train interactions: {len(train_df)} | Test interactions: {len(test_df)}")

Users: 1760 | Items: 8618
Train interactions: 86480 | Test interactions: 30253


## 1b. Sampling Strategy — Reduce to ML-100K Scale

The full filtered H&M dataset has **1,760 users** and **8,618 items** — ~5× more items than ML-100K.
This cell reduces it to a comparable scale using **stratified user sampling**:

1. Bin users into quartiles by interaction count (preserves heterogeneous data-size distribution)
2. Sample  users proportionally across quartiles
3. Retain only items touched by sampled users in training (no leakage)
4. Drop users with fewer than  in the induced train set

|  | Scale | Use case |
|---|---|---|
| 300 | Fast debug | Topology sanity check |
| 500 | **ML-100K comparable** | Main experiment |
| 1000 | Medium | Larger-scale comparison |
| 1760 | Full | Full appendix replication |

In [9]:
# ── Sampling configuration ───────────────────────────────────────────────────
TARGET_USERS           = 1000   # set to 1760 to use the full dataset
MIN_TRAIN_INTERACTIONS = 20    # same floor as ML-100K
N_QUARTILES            = 10
SAMPLED_DATA_DIR       = "hm_sampled"  # folder where CSVs are saved

import os

def sample_hm_dataset(train_df, test_df,
                      target_users=500, min_train_interactions=20,
                      seed=42, n_quartiles=4,
                      user_col="customer_id", item_col="product_code",
                      target_col="bought"):
    """
    Stratified user sampling -> induced item sub-graph.
    Returns re-indexed (train_sub, test_sub, meta).
    """
    rng = np.random.default_rng(seed)

    # Step 1: stratify users by train interaction count
    user_counts = (
        train_df.groupby(user_col)[item_col]
        .count().rename("n_interactions").reset_index()
    )
    user_counts["quartile"] = pd.qcut(
        user_counts["n_interactions"], q=n_quartiles,
        labels=False, duplicates="drop"
    )

    # Step 2: proportional sampling within each quartile
    total_users = len(user_counts)
    sampled_rows = []
    for q, grp in user_counts.groupby("quartile"):
        n_q = max(1, round(target_users * len(grp) / total_users))
        n_q = min(n_q, len(grp))
        chosen = grp.sample(n=n_q, random_state=int(rng.integers(1e6)))
        sampled_rows.append(chosen)
    sampled_users = pd.concat(sampled_rows)[user_col].unique()
    if len(sampled_users) > target_users:
        sampled_users = rng.choice(sampled_users, size=target_users, replace=False)

    # Step 3: induce item set from sampled users train interactions only
    train_sub = train_df[train_df[user_col].isin(sampled_users)].copy()
    retained_items = train_sub[item_col].unique()

    # Step 4: filter test to (sampled user, retained item) pairs
    test_sub = test_df[
        test_df[user_col].isin(sampled_users) &
        test_df[item_col].isin(retained_items)
    ].copy()

    # Step 5: drop users with too few train interactions
    sufficient = (
        train_sub.groupby(user_col)[item_col].count()
        .pipe(lambda s: s[s >= min_train_interactions]).index
    )
    train_sub = train_sub[train_sub[user_col].isin(sufficient)]
    test_sub  = test_sub[test_sub[user_col].isin(sufficient)]
    retained_items = train_sub[item_col].unique()
    test_sub = test_sub[test_sub[item_col].isin(retained_items)]

    # Re-index to 0-based contiguous integers
    user2idx = {u: i for i, u in enumerate(train_sub[user_col].unique())}
    item2idx = {a: i for i, a in enumerate(retained_items)}
    for df_ in [train_sub, test_sub]:
        df_[user_col] = df_[user_col].map(user2idx)
        df_[item_col] = df_[item_col].map(item2idx)
        df_.dropna(subset=[user_col, item_col], inplace=True)
        df_[[user_col, item_col]] = df_[[user_col, item_col]].astype(int)
    train_sub.reset_index(drop=True, inplace=True)
    test_sub.reset_index(drop=True, inplace=True)

    meta = {
        "n_users": train_sub[user_col].nunique(),
        "n_items": train_sub[item_col].nunique(),
        "n_train": len(train_sub),
        "n_test":  len(test_sub),
    }
    return train_sub, test_sub, meta


# ── Save-or-load pattern ──────────────────────────────────────────────────────
# Cache filename encodes the key parameters so different configs
# are stored as separate files and never mixed up.
cache_tag   = f"u{TARGET_USERS}_m{MIN_TRAIN_INTERACTIONS}_s{SEED}"
train_path  = os.path.join(SAMPLED_DATA_DIR, f"train_{cache_tag}.csv")
val_path    = os.path.join(SAMPLED_DATA_DIR, f"val_{cache_tag}.csv")
test_path   = os.path.join(SAMPLED_DATA_DIR, f"test_{cache_tag}.csv")
meta_path   = os.path.join(SAMPLED_DATA_DIR, f"meta_{cache_tag}.csv")

if all(os.path.exists(p) for p in [train_path, val_path, test_path, meta_path]):
    # ── Fast path: reload from disk ───────────────────────────────────────────
    print(f"Loading cached sampled dataset from {SAMPLED_DATA_DIR}/")
    train_df = pd.read_csv(train_path)
    val_df   = pd.read_csv(val_path)
    test_df  = pd.read_csv(test_path)
    meta_df  = pd.read_csv(meta_path)
    N_USERS  = int(meta_df.loc[meta_df["key"] == "n_users", "value"].iloc[0])
    N_ITEMS  = int(meta_df.loc[meta_df["key"] == "n_items", "value"].iloc[0])
    print(f"  Loaded: {N_USERS} users | {N_ITEMS} items")
    print(f"  Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

else:
    # ── Slow path: sample, split, then save ───────────────────────────────────
    print("Sampled dataset not found — running sampling pipeline...")

    train_df, test_df, meta = sample_hm_dataset(
        train_df, test_df,
        target_users=TARGET_USERS,
        min_train_interactions=MIN_TRAIN_INTERACTIONS,
        seed=SEED,
    )
    N_USERS = meta["n_users"]
    N_ITEMS = meta["n_items"]

    # Val split (80/20 of sampled train)
    from sklearn.model_selection import train_test_split
    train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=SEED)
    train_df = train_df.reset_index(drop=True)
    val_df   = val_df.reset_index(drop=True)

    # Save
    os.makedirs(SAMPLED_DATA_DIR, exist_ok=True)
    train_df.to_csv(train_path, index=False)
    val_df.to_csv(val_path,   index=False)
    test_df.to_csv(test_path,  index=False)
    pd.DataFrame([
        {"key": "n_users", "value": N_USERS},
        {"key": "n_items", "value": N_ITEMS},
        {"key": "n_train", "value": meta["n_train"]},
        {"key": "n_test",  "value": meta["n_test"]},
        {"key": "target_users",           "value": TARGET_USERS},
        {"key": "min_train_interactions",  "value": MIN_TRAIN_INTERACTIONS},
        {"key": "seed",                    "value": SEED},
    ]).to_csv(meta_path, index=False)

    density = meta["n_train"] / (N_USERS * N_ITEMS) * 100
    print(f"  Sampled & saved to {SAMPLED_DATA_DIR}/")
    print(f"  Users   : {N_USERS}")
    print(f"  Items   : {N_ITEMS}")
    print(f"  Train   : {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
    print(f"  Density : {density:.4f}%")


Sampled dataset not found — running sampling pipeline...
  Sampled & saved to hm_sampled/
  Users   : 948
  Items   : 7418
  Train   : 38708 | Val: 9677 | Test: 15696
  Density : 0.6880%


## 2. Dataset & DataLoader

## 3. MF Model